<a href="https://colab.research.google.com/github/ihstepura/publicgenai/blob/main/TASK15_credit_card_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Task 15
#Key Findings
# DistilBERT dominates classification, 50% accuracy on policy_category (21 classes) vs DistilGPT2's 1.25%.
# Encoder architectures with classification heads are inherently better suited for fixed-label classification.
#DistilGPT2 struggled with small data: With only 60 training samples, DistilGPT2 couldn't learn to generate meaningful category names or resolution text, it output garbled tokens.
#This model would benefit greatly from more training data and/or a larger model.
#Resolution is inherently harder: 78 unique resolutions in 80 records makes this nearly a unique-per-record problem. DistilBERT still achieved BLEU-1 of 0.265 by mapping to the nearest resolution class.
# Data volume matters: Both models are limited by the tiny training set (60 records, some classes with only 1 example). With more data, DistilGPT2 would likely improve significantly for the generation task.

# Credit Card Complaint Fine-Tuning Pipeline

Fine-tunes **DistilBERT** and **DistilGPT2** on two tasks:
- **Task A**: complaint → policy_category (classification, 21 classes)
- **Task B**: complaint → resolution (classification / generation)

**Training**: 60 records (stratified sample)  
**Evaluation**: all 80 records

> ⚠️ **IMPORTANT**: This notebook is designed for **Google Colab with T4 GPU**.  
> Go to **Runtime → Change runtime type → T4 GPU** before running.

**Data**: `credit_card_qa.csv` — also available at https://huggingface.co/datasets/priyaannamani/credit_card_qa

## Step 0: Install Dependencies

In [ ]:
!pip install -qU torch transformers datasets accelerate scikit-learn pandas matplotlib seaborn

## Step 1: Imports & Setup

In [ ]:
import os
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

## Step 2: Data Loading & Preprocessing

Load the credit card complaint dataset. Uses local CSV if available, otherwise loads from HuggingFace.  
Stratified split: **60 records** for training, **all 80** for evaluation.

In [ ]:
# Try local file first, then HuggingFace
CSV_PATHS = [
    'credit_card_qa.csv',
    'OneDrive_1_23-01-2026/credit_card_qa.csv',
    'credit_card_aa.csv',
]

df = None
for path in CSV_PATHS:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'📂 Loaded data from: {path}')
        break

if df is None:
    try:
        from datasets import load_dataset
        ds = load_dataset('priyaannamani/credit_card_qa', split='train')
        df = ds.to_pandas()
        print('📂 Loaded data from HuggingFace: priyaannamani/credit_card_qa')
    except Exception as e:
        raise FileNotFoundError(f'Could not find CSV or load from HuggingFace. Error: {e}')

print(f'   Total records: {len(df)}')
print(f'   Columns: {list(df.columns)}')
print(f'   Unique policy_category values: {df["policy_category"].nunique()}')
print(f'   Unique resolution values: {df["resolution"].nunique()}')
df.head()

## Step 3: Label Encoding & Train/Eval Split

In [ ]:
# --- Label Encoding for policy_category ---
le_category = LabelEncoder()
df['category_label'] = le_category.fit_transform(df['policy_category'])
NUM_CATEGORIES = len(le_category.classes_)
print(f'Policy categories ({NUM_CATEGORIES}):')
for i, cat in enumerate(le_category.classes_):
    count = (df['category_label'] == i).sum()
    print(f'  [{i}] {cat} ({count} records)')

# --- Label Encoding for resolution ---
le_resolution = LabelEncoder()
df['resolution_label'] = le_resolution.fit_transform(df['resolution'])
NUM_RESOLUTIONS = len(le_resolution.classes_)
print(f'\nResolution classes: {NUM_RESOLUTIONS}')

# --- Stratified Split: 60 train, evaluate on ALL 80 ---
train_df, _ = train_test_split(
    df, train_size=60, random_state=SEED, stratify=df['policy_category']
)
eval_df = df.copy()  # Evaluate on ALL 80 records

print(f'\nTraining samples: {len(train_df)}')
print(f'Evaluation samples: {len(eval_df)} (all records)')

## Step 4: Custom Dataset Classes

- `TextClassificationDataset` — for DistilBERT sequence classification
- `CausalLMDataset` — for DistilGPT2 causal language modeling with masked prompt loss

In [ ]:
class TextClassificationDataset(Dataset):
    '''Dataset for DistilBERT sequence classification.'''
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }


class CausalLMDataset(Dataset):
    '''Dataset for DistilGPT2 causal language modeling (prompt -> completion).'''
    def __init__(self, prompts, completions, tokenizer, max_length=256):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.input_ids_list = []
        self.attention_mask_list = []
        self.labels_list = []

        for prompt, completion in zip(prompts, completions):
            full_text = prompt + completion + tokenizer.eos_token
            encoding = tokenizer(
                full_text,
                truncation=True,
                max_length=max_length,
                padding='max_length',
                return_tensors='pt',
            )
            input_ids = encoding['input_ids'].squeeze()
            attention_mask = encoding['attention_mask'].squeeze()

            # Create labels: mask the prompt tokens with -100
            prompt_encoding = tokenizer(
                prompt, truncation=True, max_length=max_length, return_tensors='pt'
            )
            prompt_len = prompt_encoding['input_ids'].shape[1]

            labels = input_ids.clone()
            labels[:prompt_len] = -100
            labels[attention_mask == 0] = -100

            self.input_ids_list.append(input_ids)
            self.attention_mask_list.append(attention_mask)
            self.labels_list.append(labels)

    def __len__(self):
        return len(self.input_ids_list)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids_list[idx],
            'attention_mask': self.attention_mask_list[idx],
            'labels': self.labels_list[idx],
        }

print('✅ Dataset classes defined.')

## Step 5: Helper Functions

Training, evaluation, and text similarity measurement functions.

In [ ]:
def compute_classification_metrics(pred):
    '''Compute metrics for HuggingFace Trainer.'''
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}


def train_distilbert_classifier(train_texts, train_labels, eval_texts, eval_labels,
                                 num_labels, task_name, epochs=5):
    '''Fine-tune DistilBERT for sequence classification.'''
    print(f'\n{"─" * 50}')
    print(f'🔧 Training DistilBERT for: {task_name}')
    print(f'{"─" * 50}')

    model_name = 'distilbert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels
    ).to(DEVICE)

    train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer)
    eval_dataset = TextClassificationDataset(eval_texts, eval_labels, tokenizer)

    output_dir = f'./results/distilbert_{task_name}'
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        logging_dir=f'./logs/distilbert_{task_name}',
        logging_steps=10,
        seed=SEED,
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_classification_metrics,
    )

    trainer.train()
    results = trainer.evaluate()
    print(f'\n📊 DistilBERT {task_name} Results:')
    for k, v in results.items():
        if k.startswith('eval_'):
            print(f'   {k}: {v:.4f}')

    predictions = trainer.predict(eval_dataset)
    pred_labels = predictions.predictions.argmax(-1)
    true_labels = eval_labels

    return {
        'model_name': 'DistilBERT',
        'task': task_name,
        'results': results,
        'pred_labels': pred_labels,
        'true_labels': true_labels,
        'model': model,
        'tokenizer': tokenizer,
    }

print('✅ DistilBERT trainer function defined.')

In [ ]:
def train_distilgpt2_generator(train_prompts, train_completions,
                                eval_prompts, eval_completions,
                                task_name, epochs=5):
    '''Fine-tune DistilGPT2 for text generation (prompt -> completion).'''
    print(f'\n{"─" * 50}')
    print(f'🔧 Training DistilGPT2 for: {task_name}')
    print(f'{"─" * 50}')

    model_name = 'distilgpt2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
    model.config.pad_token_id = tokenizer.eos_token_id

    train_dataset = CausalLMDataset(train_prompts, train_completions, tokenizer, max_length=256)

    output_dir = f'./results/distilgpt2_{task_name}'
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=5e-5,
        weight_decay=0.01,
        logging_dir=f'./logs/distilgpt2_{task_name}',
        logging_steps=10,
        seed=SEED,
        report_to='none',
        save_strategy='epoch',
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )

    trainer.train()

    print(f'\n📝 Generating predictions for {task_name}...')
    model.eval()
    generated_texts = []

    for prompt in eval_prompts:
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=200).to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = outputs[0][inputs['input_ids'].shape[1]:]
        text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        generated_texts.append(text)

    return {
        'model_name': 'DistilGPT2',
        'task': task_name,
        'generated_texts': generated_texts,
        'reference_texts': eval_completions,
        'model': model,
        'tokenizer': tokenizer,
    }

print('✅ DistilGPT2 trainer function defined.')

In [ ]:
def evaluate_gpt2_as_classifier(generated_texts, reference_texts, label_encoder, task_name):
    '''Evaluate GPT2 generation output as classification by matching to known labels.'''
    known_labels = list(label_encoder.classes_)
    pred_indices = []
    true_indices = []
    exact_matches = 0

    for gen, ref in zip(generated_texts, reference_texts):
        ref_clean = ref.strip()
        gen_clean = gen.strip().split('\n')[0].strip()

        true_idx = label_encoder.transform([ref_clean])[0] if ref_clean in known_labels else -1
        true_indices.append(true_idx)

        if gen_clean in known_labels:
            pred_idx = label_encoder.transform([gen_clean])[0]
            exact_matches += 1
        else:
            best_match = None
            best_score = 0
            for label in known_labels:
                gen_lower = gen_clean.lower()
                label_lower = label.lower()
                if label_lower in gen_lower or gen_lower in label_lower:
                    score = len(label_lower)
                    if score > best_score:
                        best_score = score
                        best_match = label
            if best_match:
                pred_idx = label_encoder.transform([best_match])[0]
            else:
                pred_idx = 0

        pred_indices.append(pred_idx)

    acc = accuracy_score(true_indices, pred_indices)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_indices, pred_indices, average='weighted', zero_division=0
    )

    print(f'\n📊 DistilGPT2 {task_name} Classification Metrics:')
    print(f'   Exact match rate: {exact_matches}/{len(generated_texts)} ({exact_matches/len(generated_texts)*100:.1f}%)')
    print(f'   Accuracy: {acc:.4f}')
    print(f'   F1 (weighted): {f1:.4f}')
    print(f'   Precision: {precision:.4f}')
    print(f'   Recall: {recall:.4f}')

    return {
        'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall,
        'exact_match_rate': exact_matches / len(generated_texts),
        'pred_labels': pred_indices, 'true_labels': true_indices,
    }


def compute_text_similarity(generated_texts, reference_texts):
    '''Compute BLEU-1 and ROUGE-L between generated and reference texts.'''
    from collections import Counter

    def simple_bleu(reference, hypothesis):
        ref_tokens = reference.lower().split()
        hyp_tokens = hypothesis.lower().split()
        if not hyp_tokens or not ref_tokens:
            return 0.0
        ref_counter = Counter(ref_tokens)
        hyp_counter = Counter(hyp_tokens)
        overlap = sum((hyp_counter & ref_counter).values())
        return overlap / len(hyp_tokens) if hyp_tokens else 0.0

    def rouge_l(reference, hypothesis):
        ref_tokens = reference.lower().split()
        hyp_tokens = hypothesis.lower().split()
        if not ref_tokens or not hyp_tokens:
            return 0.0
        m, n = len(ref_tokens), len(hyp_tokens)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if ref_tokens[i-1] == hyp_tokens[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        lcs = dp[m][n]
        prec = lcs / n if n > 0 else 0
        rec = lcs / m if m > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        return f1

    bleu_scores = [simple_bleu(ref, gen) for gen, ref in zip(generated_texts, reference_texts)]
    rouge_scores = [rouge_l(ref, gen) for gen, ref in zip(generated_texts, reference_texts)]

    return {
        'bleu_1': np.mean(bleu_scores), 'rouge_l': np.mean(rouge_scores),
        'bleu_scores': bleu_scores, 'rouge_scores': rouge_scores,
    }

print('✅ Evaluation helper functions defined.')

## Step 6: Task A — Complaint → Policy Category

Fine-tune both **DistilBERT** (classification) and **DistilGPT2** (generation) to predict the policy category from a complaint.

In [ ]:
train_texts_a = train_df['complaint'].tolist()
train_labels_a = train_df['category_label'].tolist()
eval_texts_a = eval_df['complaint'].tolist()
eval_labels_a = eval_df['category_label'].tolist()
print(f'Task A data prepared: {len(train_texts_a)} train, {len(eval_texts_a)} eval')

### 6a. DistilBERT — Policy Category Classification

In [ ]:
bert_cat_results = train_distilbert_classifier(
    train_texts_a, train_labels_a,
    eval_texts_a, eval_labels_a,
    num_labels=NUM_CATEGORIES,
    task_name='policy_category',
    epochs=5,
)

print('\n📋 DistilBERT Policy Category — Full Classification Report:')
print(classification_report(
    bert_cat_results['true_labels'],
    bert_cat_results['pred_labels'],
    target_names=le_category.classes_,
    zero_division=0,
))

### 6b. DistilGPT2 — Policy Category Generation

In [ ]:
train_prompts_cat = [
    f'Classify the following credit card complaint into a policy category.\n\nComplaint: {t}\n\nCategory: '
    for t in train_df['complaint']
]
train_completions_cat = train_df['policy_category'].tolist()
eval_prompts_cat = [
    f'Classify the following credit card complaint into a policy category.\n\nComplaint: {t}\n\nCategory: '
    for t in eval_df['complaint']
]
eval_completions_cat = eval_df['policy_category'].tolist()

gpt2_cat_results = train_distilgpt2_generator(
    train_prompts_cat, train_completions_cat,
    eval_prompts_cat, eval_completions_cat,
    task_name='policy_category',
    epochs=5,
)

gpt2_cat_metrics = evaluate_gpt2_as_classifier(
    gpt2_cat_results['generated_texts'],
    gpt2_cat_results['reference_texts'],
    le_category,
    'policy_category',
)

print('\n📝 Sample DistilGPT2 Policy Category Predictions:')
for i in range(min(5, len(eval_texts_a))):
    print(f'\n  Complaint: {eval_texts_a[i][:100]}...')
    print(f'  Expected:  {eval_completions_cat[i]}')
    print(f'  Generated: {gpt2_cat_results["generated_texts"][i][:100]}')

## Step 7: Task B — Complaint → Resolution

Fine-tune both **DistilBERT** (classification) and **DistilGPT2** (generation) to predict the resolution from a complaint.

In [ ]:
train_texts_b = train_df['complaint'].tolist()
train_labels_b = train_df['resolution_label'].tolist()
eval_texts_b = eval_df['complaint'].tolist()
eval_labels_b = eval_df['resolution_label'].tolist()
print(f'Task B data prepared: {len(train_texts_b)} train, {len(eval_texts_b)} eval')

### 7a. DistilBERT — Resolution Classification

In [ ]:
bert_res_results = train_distilbert_classifier(
    train_texts_b, train_labels_b,
    eval_texts_b, eval_labels_b,
    num_labels=NUM_RESOLUTIONS,
    task_name='resolution',
    epochs=5,
)

print('\n📋 DistilBERT Resolution — Full Classification Report:')
print(classification_report(
    bert_res_results['true_labels'],
    bert_res_results['pred_labels'],
    target_names=[le_resolution.classes_[i][:50] for i in range(NUM_RESOLUTIONS)],
    zero_division=0,
))

# Text similarity for DistilBERT resolution predictions
bert_res_pred_texts = [le_resolution.classes_[p] for p in bert_res_results['pred_labels']]
bert_res_true_texts = [le_resolution.classes_[t] for t in bert_res_results['true_labels']]
bert_res_text_metrics = compute_text_similarity(bert_res_pred_texts, bert_res_true_texts)
print(f'\n📊 DistilBERT Resolution Text Similarity:')
print(f'   BLEU-1: {bert_res_text_metrics["bleu_1"]:.4f}')
print(f'   ROUGE-L: {bert_res_text_metrics["rouge_l"]:.4f}')

### 7b. DistilGPT2 — Resolution Generation

In [ ]:
train_prompts_res = [
    f'You are a credit card customer service agent. Based on the complaint, provide the appropriate resolution.\n\nComplaint: {t}\n\nResolution: '
    for t in train_df['complaint']
]
train_completions_res = train_df['resolution'].tolist()
eval_prompts_res = [
    f'You are a credit card customer service agent. Based on the complaint, provide the appropriate resolution.\n\nComplaint: {t}\n\nResolution: '
    for t in eval_df['complaint']
]
eval_completions_res = eval_df['resolution'].tolist()

gpt2_res_results = train_distilgpt2_generator(
    train_prompts_res, train_completions_res,
    eval_prompts_res, eval_completions_res,
    task_name='resolution',
    epochs=5,
)

gpt2_res_text_metrics = compute_text_similarity(
    gpt2_res_results['generated_texts'],
    gpt2_res_results['reference_texts'],
)
print(f'\n📊 DistilGPT2 Resolution Text Similarity:')
print(f'   BLEU-1: {gpt2_res_text_metrics["bleu_1"]:.4f}')
print(f'   ROUGE-L: {gpt2_res_text_metrics["rouge_l"]:.4f}')

gpt2_res_class_metrics = evaluate_gpt2_as_classifier(
    gpt2_res_results['generated_texts'],
    gpt2_res_results['reference_texts'],
    le_resolution,
    'resolution',
)

print('\n📝 Sample DistilGPT2 Resolution Predictions:')
for i in range(min(5, len(eval_texts_b))):
    print(f'\n  Complaint: {eval_texts_b[i][:100]}...')
    print(f'  Expected:  {eval_completions_res[i][:100]}...')
    print(f'  Generated: {gpt2_res_results["generated_texts"][i][:100]}')

## Step 8: Comparative Analysis

Side-by-side comparison of all 4 model-task combinations.

In [ ]:
comparison_data = {
    'Model': ['DistilBERT', 'DistilGPT2', 'DistilBERT', 'DistilGPT2'],
    'Task': ['Policy Category', 'Policy Category', 'Resolution', 'Resolution'],
    'Approach': ['Classification', 'Generation→Match', 'Classification', 'Generation'],
    'Accuracy': [
        bert_cat_results['results'].get('eval_accuracy', 0),
        gpt2_cat_metrics['accuracy'],
        bert_res_results['results'].get('eval_accuracy', 0),
        gpt2_res_class_metrics['accuracy'],
    ],
    'F1 (Weighted)': [
        bert_cat_results['results'].get('eval_f1', 0),
        gpt2_cat_metrics['f1'],
        bert_res_results['results'].get('eval_f1', 0),
        gpt2_res_class_metrics['f1'],
    ],
    'Precision': [
        bert_cat_results['results'].get('eval_precision', 0),
        gpt2_cat_metrics['precision'],
        bert_res_results['results'].get('eval_precision', 0),
        gpt2_res_class_metrics['precision'],
    ],
    'Recall': [
        bert_cat_results['results'].get('eval_recall', 0),
        gpt2_cat_metrics['recall'],
        bert_res_results['results'].get('eval_recall', 0),
        gpt2_res_class_metrics['recall'],
    ],
    'BLEU-1': [
        'N/A', 'N/A',
        f'{bert_res_text_metrics["bleu_1"]:.4f}',
        f'{gpt2_res_text_metrics["bleu_1"]:.4f}',
    ],
    'ROUGE-L': [
        'N/A', 'N/A',
        f'{bert_res_text_metrics["rouge_l"]:.4f}',
        f'{gpt2_res_text_metrics["rouge_l"]:.4f}',
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print('=' * 80)
print('RESULTS COMPARISON TABLE')
print('=' * 80)
print(comparison_df.to_string(index=False))
print('=' * 80)
comparison_df

## Step 9: Visualizations

Bar charts, heatmaps, and confusion matrices comparing all model-task combinations.

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})
os.makedirs('./results/plots', exist_ok=True)

# --- Figure 1: Accuracy & F1 Comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

task_a_models = ['DistilBERT', 'DistilGPT2']
task_a_acc = [
    bert_cat_results['results'].get('eval_accuracy', 0),
    gpt2_cat_metrics['accuracy'],
]
task_a_f1 = [
    bert_cat_results['results'].get('eval_f1', 0),
    gpt2_cat_metrics['f1'],
]

x = np.arange(len(task_a_models))
width = 0.35
bars1 = axes[0].bar(x - width/2, task_a_acc, width, label='Accuracy', color='#4C72B0')
bars2 = axes[0].bar(x + width/2, task_a_f1, width, label='F1 Score', color='#DD8452')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Score')
axes[0].set_title('Task A: Policy Category Classification')
axes[0].set_xticks(x)
axes[0].set_xticklabels(task_a_models)
axes[0].set_ylim(0, 1.1)
axes[0].legend()
for bar in bars1:
    axes[0].annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=10)
for bar in bars2:
    axes[0].annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=10)

# Task B: Resolution
task_b_acc = [
    bert_res_results['results'].get('eval_accuracy', 0),
    gpt2_res_class_metrics['accuracy'],
]
task_b_f1 = [
    bert_res_results['results'].get('eval_f1', 0),
    gpt2_res_class_metrics['f1'],
]

bars3 = axes[1].bar(x - width/2, task_b_acc, width, label='Accuracy', color='#4C72B0')
bars4 = axes[1].bar(x + width/2, task_b_f1, width, label='F1 Score', color='#DD8452')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Score')
axes[1].set_title('Task B: Resolution Prediction')
axes[1].set_xticks(x)
axes[1].set_xticklabels(task_a_models)
axes[1].set_ylim(0, 1.1)
axes[1].legend()
for bar in bars3:
    axes[1].annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=10)
for bar in bars4:
    axes[1].annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('./results/plots/accuracy_f1_comparison.png', bbox_inches='tight')
plt.show()
print('✅ Saved: results/plots/accuracy_f1_comparison.png')

In [ ]:
# --- Figure 2: Text Similarity Metrics for Resolution Task ---
fig, ax = plt.subplots(figsize=(10, 6))

models = ['DistilBERT', 'DistilGPT2']
bleu_vals = [bert_res_text_metrics['bleu_1'], gpt2_res_text_metrics['bleu_1']]
rouge_vals = [bert_res_text_metrics['rouge_l'], gpt2_res_text_metrics['rouge_l']]

x = np.arange(len(models))
width = 0.35
bars1 = ax.bar(x - width/2, bleu_vals, width, label='BLEU-1', color='#55A868')
bars2 = ax.bar(x + width/2, rouge_vals, width, label='ROUGE-L', color='#C44E52')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Resolution Task: Text Similarity Metrics')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.1)
ax.legend()
for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('./results/plots/resolution_text_similarity.png', bbox_inches='tight')
plt.show()
print('✅ Saved: results/plots/resolution_text_similarity.png')

In [ ]:
# --- Figure 3: All Metrics Heatmap ---
fig, ax = plt.subplots(figsize=(12, 5))

heatmap_data = pd.DataFrame({
    'DistilBERT\nPolicy Cat.': [
        bert_cat_results['results'].get('eval_accuracy', 0),
        bert_cat_results['results'].get('eval_f1', 0),
        bert_cat_results['results'].get('eval_precision', 0),
        bert_cat_results['results'].get('eval_recall', 0),
    ],
    'DistilGPT2\nPolicy Cat.': [
        gpt2_cat_metrics['accuracy'],
        gpt2_cat_metrics['f1'],
        gpt2_cat_metrics['precision'],
        gpt2_cat_metrics['recall'],
    ],
    'DistilBERT\nResolution': [
        bert_res_results['results'].get('eval_accuracy', 0),
        bert_res_results['results'].get('eval_f1', 0),
        bert_res_results['results'].get('eval_precision', 0),
        bert_res_results['results'].get('eval_recall', 0),
    ],
    'DistilGPT2\nResolution': [
        gpt2_res_class_metrics['accuracy'],
        gpt2_res_class_metrics['f1'],
        gpt2_res_class_metrics['precision'],
        gpt2_res_class_metrics['recall'],
    ],
}, index=['Accuracy', 'F1', 'Precision', 'Recall'])

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5,
            vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'Score'})
ax.set_title('Performance Comparison: All Models × All Tasks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/plots/performance_heatmap.png', bbox_inches='tight')
plt.show()
print('✅ Saved: results/plots/performance_heatmap.png')

In [ ]:
# --- Figure 4: Confusion Matrices for Policy Category ---
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# DistilBERT
cm_bert = confusion_matrix(bert_cat_results['true_labels'], bert_cat_results['pred_labels'])
used_labels = sorted(set(bert_cat_results['true_labels']) | set(bert_cat_results['pred_labels']))
label_names = [le_category.classes_[i][:20] for i in used_labels]
cm_bert_filtered = cm_bert[np.ix_(used_labels, used_labels)]

sns.heatmap(cm_bert_filtered, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=label_names, yticklabels=label_names)
axes[0].set_title('DistilBERT: Policy Category\nConfusion Matrix', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# DistilGPT2
cm_gpt2 = confusion_matrix(gpt2_cat_metrics['true_labels'], gpt2_cat_metrics['pred_labels'])
used_labels_g = sorted(set(gpt2_cat_metrics['true_labels']) | set(gpt2_cat_metrics['pred_labels']))
label_names_g = [le_category.classes_[i][:20] if i < len(le_category.classes_) else '?' for i in used_labels_g]
cm_gpt2_filtered = cm_gpt2[np.ix_(used_labels_g, used_labels_g)] if len(used_labels_g) > 0 else cm_gpt2

sns.heatmap(cm_gpt2_filtered, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=label_names_g, yticklabels=label_names_g)
axes[1].set_title('DistilGPT2: Policy Category\nConfusion Matrix', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('./results/plots/confusion_matrices.png', bbox_inches='tight')
plt.show()
print('✅ Saved: results/plots/confusion_matrices.png')

## Step 10: Final Analysis Summary

In [ ]:

print('\n✅ Fine-tuning pipeline complete!')
print('   Results saved to: ./results/plots/')
print('   Comparison table printed above.')